# Chapter 09 - Numerical Feature Engineering

Raw numerical features rarely arrive in a form that is optimal for machine learning models.
Values may live on wildly different scales, contain outliers, or follow skewed distributions.
This notebook covers the core techniques for transforming numerical features into shapes that
algorithms can learn from effectively.

**Topics covered:**
- Why feature engineering matters
- Feature scaling: StandardScaler, MinMaxScaler, RobustScaler
- When scaling is necessary (and when it is not)
- Handling outliers: IQR method, clipping, log transformation
- Power transforms: `np.log1p`, Box-Cox, Yeo-Johnson
- Binning / discretization with KBinsDiscretizer
- Creating interaction features
- Visualizing distributions before and after transformations

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    PowerTransformer,
    KBinsDiscretizer,
    PolynomialFeatures,
)
from sklearn.datasets import fetch_california_housing

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True

## Why Feature Engineering Matters

There is a well-known saying in machine learning: **garbage in, garbage out**. The quality of
your features directly determines the ceiling of your model's performance. A sophisticated
algorithm trained on poorly prepared features will almost always lose to a simpler model
trained on well-engineered features.

Feature engineering for numerical data typically involves:
1. **Scaling** values so that features are comparable
2. **Reducing the influence of outliers** that can dominate distance calculations
3. **Reshaping distributions** so that algorithms that assume normality perform better
4. **Creating new features** by combining existing ones (interactions, polynomials)

## Sample Data

We will use the California Housing dataset throughout this notebook. It has a good mix of
feature scales, skewed distributions, and outliers.

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe().round(2)

> Notice how `MedInc` ranges roughly 0-15 while `Population` ranges 3-35,000.
> Without scaling, any distance-based algorithm would be dominated by `Population`.

## Feature Scaling

Scaling brings features onto a comparable range. Scikit-learn provides several scalers,
each suited to different situations.

### StandardScaler (Z-score Normalization)

Transforms each feature to have **mean = 0** and **standard deviation = 1**:

$$z = \frac{x - \mu}{\sigma}$$

This is the most common scaler and works well when features are approximately Gaussian.

In [ ]:
scaler = StandardScaler()
X_std = scaler.fit_transform(df[['MedInc', 'Population']])

print('Before scaling:')
print(f'  MedInc     -> mean={df["MedInc"].mean():.2f}, std={df["MedInc"].std():.2f}')
print(f'  Population -> mean={df["Population"].mean():.2f}, std={df["Population"].std():.2f}')
print()
print('After StandardScaler:')
print(f'  MedInc     -> mean={X_std[:, 0].mean():.2f}, std={X_std[:, 0].std():.2f}')
print(f'  Population -> mean={X_std[:, 1].mean():.2f}, std={X_std[:, 1].std():.2f}')

### MinMaxScaler

Scales each feature to a fixed range, usually **[0, 1]**:

$$x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

Useful when you need bounded values (e.g., image pixel inputs, neural networks).
However, it is sensitive to outliers since they determine the min and max.

In [ ]:
minmax = MinMaxScaler()
X_mm = minmax.fit_transform(df[['MedInc', 'Population']])

print('After MinMaxScaler:')
print(f'  MedInc     -> min={X_mm[:, 0].min():.2f}, max={X_mm[:, 0].max():.2f}')
print(f'  Population -> min={X_mm[:, 1].min():.2f}, max={X_mm[:, 1].max():.2f}')

### RobustScaler

Uses the **median** and **interquartile range (IQR)** instead of mean and standard deviation:

$$x_{\text{scaled}} = \frac{x - \text{median}}{\text{IQR}}$$

This makes it **robust to outliers** — the extreme values do not heavily influence the
scaling parameters.

In [ ]:
robust = RobustScaler()
X_rb = robust.fit_transform(df[['MedInc', 'Population']])

print('After RobustScaler:')
print(f'  MedInc     -> median={np.median(X_rb[:, 0]):.2f}, IQR range=[{np.percentile(X_rb[:, 0], 25):.2f}, {np.percentile(X_rb[:, 0], 75):.2f}]')
print(f'  Population -> median={np.median(X_rb[:, 1]):.2f}, IQR range=[{np.percentile(X_rb[:, 1], 25):.2f}, {np.percentile(X_rb[:, 1], 75):.2f}]')

### Comparing All Three Scalers Visually

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
feature = 'MedInc'

axes[0].hist(df[feature], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title(f'Original {feature}')

axes[1].hist(StandardScaler().fit_transform(df[[feature]]), bins=50, edgecolor='black', alpha=0.7)
axes[1].set_title('StandardScaler')

axes[2].hist(MinMaxScaler().fit_transform(df[[feature]]), bins=50, edgecolor='black', alpha=0.7)
axes[2].set_title('MinMaxScaler')

axes[3].hist(RobustScaler().fit_transform(df[[feature]]), bins=50, edgecolor='black', alpha=0.7)
axes[3].set_title('RobustScaler')

for ax in axes:
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

> **Key observation:** Scaling changes the range and center of the data but does **not**
> change the shape of the distribution. The histograms all have the same shape — only
> the axis labels differ.

## When to Scale (and When Not To)

| Algorithm family | Needs scaling? | Why |
|---|---|---|
| **KNN, SVM, Logistic Regression** | Yes | Distance or gradient-based; large features dominate |
| **Neural Networks** | Yes | Gradient descent converges faster with normalized inputs |
| **PCA** | Yes | Maximizes variance; unscaled features with large range dominate |
| **Decision Trees, Random Forest, Gradient Boosting** | No | Split-based; invariant to monotonic transformations |

> **Rule of thumb:** if the algorithm uses distances or gradients, scale your features.
> If it uses splits (trees), scaling is unnecessary.

## Handling Outliers

Outliers can distort scaling, inflate variance, and mislead models. There are several
strategies for dealing with them.

### Detecting Outliers with the IQR Method

The interquartile range (IQR) is the range between the 25th and 75th percentiles.
Values beyond **1.5 * IQR** from the quartiles are considered outliers.

In [ ]:
feature = 'Population'
Q1 = df[feature].quantile(0.25)
Q3 = df[feature].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df[feature] < lower_bound) | (df[feature] > upper_bound)]
print(f'Feature: {feature}')
print(f'Q1={Q1:.0f}, Q3={Q3:.0f}, IQR={IQR:.0f}')
print(f'Bounds: [{lower_bound:.0f}, {upper_bound:.0f}]')
print(f'Outliers: {len(outliers)} out of {len(df)} rows ({len(outliers)/len(df)*100:.1f}%)')

### Clipping (Winsorization)

Instead of removing outliers, we can **clip** them to a maximum/minimum value.
This preserves the number of samples while limiting extreme influence.

In [ ]:
population_clipped = df['Population'].clip(lower=lower_bound, upper=upper_bound)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['Population'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(upper_bound, color='red', linestyle='--', label=f'Upper bound ({upper_bound:.0f})')
axes[0].set_title('Original Population')
axes[0].legend()

axes[1].hist(population_clipped, bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_title('Clipped Population')

plt.tight_layout()
plt.show()

### Log Transformation for Right-Skewed Data

Many real-world features (income, population, prices) are right-skewed.
A log transformation compresses the long tail and can make the distribution more symmetric.

In [ ]:
# np.log1p computes log(1 + x), which handles zero values gracefully
population_log = np.log1p(df['Population'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['Population'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Original Population')
axes[0].set_xlabel('Population')

axes[1].hist(population_log, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_title('After log1p Transform')
axes[1].set_xlabel('log(1 + Population)')

plt.tight_layout()
plt.show()

print(f'Skewness before: {df["Population"].skew():.2f}')
print(f'Skewness after:  {population_log.skew():.2f}')

## Power Transforms

Power transforms are a family of parametric transformations that aim to make data
more Gaussian-like. Scikit-learn's `PowerTransformer` supports two methods:

- **Box-Cox:** Only works with **strictly positive** data (`x > 0`)
- **Yeo-Johnson:** Works with **any** real-valued data (including zero and negative values)

Both automatically find the optimal transformation parameter.

In [ ]:
# Yeo-Johnson works on any data
pt_yj = PowerTransformer(method='yeo-johnson')
pop_yj = pt_yj.fit_transform(df[['Population']])

# Box-Cox requires strictly positive data (Population is always positive here)
pt_bc = PowerTransformer(method='box-cox')
pop_bc = pt_bc.fit_transform(df[['Population']])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['Population'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Original')

axes[1].hist(pop_yj, bins=50, edgecolor='black', alpha=0.7, color='purple')
axes[1].set_title(f'Yeo-Johnson (lambda={pt_yj.lambdas_[0]:.3f})')

axes[2].hist(pop_bc, bins=50, edgecolor='black', alpha=0.7, color='teal')
axes[2].set_title(f'Box-Cox (lambda={pt_bc.lambdas_[0]:.3f})')

plt.tight_layout()
plt.show()

In [ ]:
# PowerTransformer also standardizes the output (mean=0, std=1) by default
print(f'Yeo-Johnson output -> mean={pop_yj.mean():.4f}, std={pop_yj.std():.4f}')
print(f'Box-Cox output     -> mean={pop_bc.mean():.4f}, std={pop_bc.std():.4f}')

> **When to use which:**
> - Use `np.log1p` for a quick, interpretable transform on positive data
> - Use **Box-Cox** when all values are strictly positive and you want the optimal lambda
> - Use **Yeo-Johnson** as the general-purpose choice (handles zeros and negatives)

## Binning / Discretization

Binning converts a continuous feature into discrete categories. This can help models
capture non-linear relationships and reduce the impact of minor measurement noise.

Scikit-learn's `KBinsDiscretizer` supports three strategies:
- `uniform`: Equal-width bins
- `quantile`: Equal-frequency bins (each bin has roughly the same number of samples)
- `kmeans`: Bin edges based on k-means clustering

In [ ]:
# Discretize MedInc into 5 quantile-based bins
kbd = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')
income_binned = kbd.fit_transform(df[['MedInc']])

# Show the bin edges
print('Bin edges:', [f'{edge:.2f}' for edge in kbd.bin_edges_[0]])
print()

# Count samples per bin
unique, counts = np.unique(income_binned, return_counts=True)
for bin_id, count in zip(unique, counts):
    print(f'  Bin {int(bin_id)}: {count} samples')

In [ ]:
# Compare uniform vs quantile binning
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['MedInc'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('Original MedInc')

for idx, strategy in enumerate(['uniform', 'quantile'], start=1):
    kbd_s = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy=strategy)
    binned = kbd_s.fit_transform(df[['MedInc']])
    axes[idx].hist(binned, bins=5, edgecolor='black', alpha=0.7, color=['green', 'orange'][idx-1])
    axes[idx].set_title(f'{strategy.title()} Binning (5 bins)')

plt.tight_layout()
plt.show()

> **Quantile binning** produces roughly equal-sized bins, which is better when the
> original distribution is skewed. **Uniform binning** can leave many bins nearly empty
> if the data is concentrated in a narrow range.

## Creating Interaction Features

Sometimes the relationship between two features is more informative than either feature
alone. For example, "rooms per household" might predict housing price better than
total rooms or total households separately.

You can create interaction features manually or use `PolynomialFeatures`.

In [ ]:
# Manual interaction features
df_eng = df.copy()
df_eng['rooms_per_household'] = df_eng['AveRooms'] / df_eng['HouseAge'].clip(lower=1)
df_eng['bedrooms_ratio'] = df_eng['AveBedrms'] / df_eng['AveRooms'].clip(lower=0.1)
df_eng['people_per_household'] = df_eng['Population'] / df_eng['AveOccup'].clip(lower=1)

print('New features:')
df_eng[['rooms_per_household', 'bedrooms_ratio', 'people_per_household']].describe().round(3)

In [ ]:
# Using PolynomialFeatures for systematic interaction generation
# degree=2, interaction_only=True gives all pairwise products (no squared terms)
sample = df[['MedInc', 'HouseAge']].head(5)
print('Original features:')
print(sample.values)
print()

poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
interactions = poly.fit_transform(sample)

print(f'Feature names: {poly.get_feature_names_out()}')
print(f'Shape: {sample.shape} -> {interactions.shape}')
print()
print('With interaction:')
print(interactions)

In [ ]:
# Full polynomial features (includes squared terms)
poly_full = PolynomialFeatures(degree=2, include_bias=False)
full_features = poly_full.fit_transform(sample)

print(f'Feature names: {poly_full.get_feature_names_out()}')
print(f'Shape: {sample.shape} -> {full_features.shape}')

> **Caution:** PolynomialFeatures with many input columns and degree > 2 creates a
> combinatorial explosion of features. For example, 10 features with degree=3 produces
> 286 output features. Always consider whether the added complexity is justified.

## Visualizing Distributions Before and After Transformation

A systematic comparison helps you choose the right transformation for each feature.
Let us compare the effect of several transforms on `AveOccup`, which has a heavy right tail.

In [ ]:
feature = 'AveOccup'
raw = df[[feature]].copy()

transforms = {
    'Original': raw.values.ravel(),
    'StandardScaler': StandardScaler().fit_transform(raw).ravel(),
    'log1p': np.log1p(raw.values).ravel(),
    'Yeo-Johnson': PowerTransformer(method='yeo-johnson').fit_transform(raw).ravel(),
    'Clipped + Scaled': StandardScaler().fit_transform(
        raw.clip(lower=raw.quantile(0.01).iloc[0], upper=raw.quantile(0.99).iloc[0])
    ).ravel(),
}

fig, axes = plt.subplots(1, len(transforms), figsize=(20, 4))

for ax, (name, values) in zip(axes, transforms.items()):
    ax.hist(values, bins=50, edgecolor='black', alpha=0.7)
    ax.set_title(name)
    ax.set_xlabel(f'skew={pd.Series(values).skew():.2f}')

plt.suptitle(f'Comparing Transforms on {feature}', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## The fit / transform Pattern

Every scikit-learn transformer follows the same API:

1. **`fit(X_train)`** — learn parameters from training data (e.g., mean, std, bin edges)
2. **`transform(X)`** — apply the learned transformation to any data
3. **`fit_transform(X_train)`** — convenience shortcut for fit + transform on training data

The critical rule: **always fit on training data only**, then use `transform` on test data.
Fitting on the full dataset causes **data leakage** — the model indirectly sees test data
statistics during training.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Correct: fit on training data, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform
X_test_scaled = scaler.transform(X_test)         # transform only (uses train statistics)

print(f'Train mean: {X_train_scaled.mean(axis=0).round(4)}')
print(f'Test mean:  {X_test_scaled.mean(axis=0).round(4)}')  # close to 0, but not exactly

> The test set mean is **close to zero** but not exactly zero — this is expected, because
> the scaler was fit on the training set. This is the correct behavior.

## Quick Reference: Choosing a Transform

| Situation | Recommended transform |
|---|---|
| Features on very different scales | StandardScaler or MinMaxScaler |
| Data contains significant outliers | RobustScaler or clip first |
| Right-skewed positive data | `np.log1p` or Box-Cox |
| Skewed data with zeros or negatives | Yeo-Johnson |
| Need bounded [0, 1] output | MinMaxScaler |
| Want to capture non-linear patterns for linear models | PolynomialFeatures or binning |
| Tree-based model | Usually no transform needed |

---

**Summary:** This notebook covered the essential techniques for preparing numerical features:
scaling (StandardScaler, MinMaxScaler, RobustScaler), outlier handling (IQR detection, clipping,
log transforms), power transforms (Box-Cox, Yeo-Johnson), binning, and interaction features.
The right transformation depends on your data distribution and the algorithm you plan to use.

**Next up:** Categorical feature encoding.